# Notebook 4.0: Sentiment Analysis & Model Validation (RQ1)

**Research Question 1 (RQ1) & Primary Scoring (`2018-01-01` to `2022-12-31`, 7 Subreddits)**

This notebook:
1. Loads sampled comments from `data/samples/2.0/` and cleans text with `apply_sentiment_cleaning()` (preserving casing, negations, emojis, and expressive punctuation).
2. Scores full-sample VADER sentiment with caching (`sentiment_vader_full`).
3. Draws a balanced stratified subset (`roberta_sample_per_subreddit` per subreddit) and runs 100% genuine `cardiffnlp/twitter-roberta-base-sentiment` scoring (with cache key `sentiment_roberta_7subs_{per_sub}per_sub_seed{seed}`).
4. Compares VADER vs. RoBERTa (`agreement_rate`, `cohen_kappa`, `macro_f1`, neutral percentages, and top-5 disagreements).
5. Exports `data/processed/sentiment_df.parquet`, RQ1 tables (`data/final/tables/`), and comparison figure (`data/final/figures/`).

In [ ]:
import sys
from pathlib import Path

nb_dir = Path.cwd()
project_root = nb_dir.parent if nb_dir.name == 'notebooks' else nb_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.modeling import set_seed
from src.modeling.cache import load_parquet_cache, save_parquet_cache
from src.modeling.load import draw_stratified_subset, load_samples
from src.modeling.sentiment import compare_models, load_roberta, score_roberta, score_vader
from src.modeling.text import apply_sentiment_cleaning
from src.modeling.tracking import log_experiment

config_path = project_root / 'config' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

seed = int(cfg['modeling']['seed'])
set_seed(seed)

samples_dir = project_root / cfg['paths']['data_samples_dir'] / '2.0'
cache_dir = project_root / cfg['modeling']['cache_dir']
exp_dir = project_root / cfg['modeling']['experiment_log_dir']
processed_dir = project_root / cfg['paths']['processed_data_dir']
final_dir = project_root / cfg['paths']['final_dir']
tables_dir = final_dir / 'tables'
fig_dir = final_dir / 'figures'
for d in (cache_dir, exp_dir, processed_dir, tables_dir, fig_dir):
    d.mkdir(parents=True, exist_ok=True)

print(f'Sentiment analysis pipeline initialized (seed={seed}).')

## Step 1: Load Comment Samples & Apply Sentiment-Safe Cleaning
Use `apply_sentiment_cleaning()` so negation, capitalization, emojis, and punctuation cues remain intact.

In [ ]:
df_comments = load_samples(
    samples_dir=samples_dir,
    subreddits=cfg['analysis']['subreddits'],
    dataset_types=['comments'],
)

start_ts = pd.Timestamp(cfg['analysis']['start_date'], tz='UTC')
end_ts = pd.Timestamp(cfg['analysis']['end_date'], tz='UTC') + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)
df_comments = (
    df_comments[(df_comments['created_utc_dt'] >= start_ts) & (df_comments['created_utc_dt'] <= end_ts)]
    .copy()
    .reset_index(drop=True)
)

text_col = 'text' if 'text' in df_comments.columns else 'body'
df_comments['cleaned_text'] = apply_sentiment_cleaning(df_comments[text_col])
df_comments = df_comments[df_comments['cleaned_text'].str.split().str.len() >= 3].reset_index(drop=True)

print(f'Ready for sentiment scoring: {len(df_comments):,} comments across {df_comments["subreddit"].nunique()} subreddits.')
print(df_comments['subreddit'].value_counts())

## Step 2: Full-Sample VADER Scoring (Cross-Check Baseline)
Load cached VADER scores (`sentiment_vader_full`) or compute VADER across the full comment corpus (`|compound| <= 0.05` neutral band) and cache to Parquet.

In [ ]:
vader_full = load_parquet_cache('sentiment_vader_full', cache_dir=cache_dir)
if vader_full is None or len(vader_full) != len(df_comments):
    print(f'Scoring VADER on full {len(df_comments):,} comments...')
    v_res = score_vader(df_comments['cleaned_text'].tolist(), neutral_band=0.05)
    df_comments['vader_compound'] = v_res['vader_compound'].values
    df_comments['vader_label'] = v_res['vader_label'].values
    vader_full = df_comments
    save_parquet_cache(vader_full, 'sentiment_vader_full', cache_dir=cache_dir)
else:
    vader_full = vader_full.reset_index(drop=True)
    print(f'Loaded cached full VADER scores ({len(vader_full):,} rows).')

print('VADER sentiment distribution (full corpus):')
print(vader_full['vader_label'].value_counts(normalize=True).round(4))

## Step 3: Balanced Stratified Subset & Genuine RoBERTa Scoring
Draw a balanced subset (`roberta_sample_per_subreddit` per subreddit) via `draw_stratified_subset()` and load or compute genuine RoBERTa probabilities (`sentiment_roberta_7subs_{per_sub}per_sub_seed{seed}`).

In [ ]:
per_sub = int(cfg['modeling']['roberta_sample_per_subreddit'])
roberta_key = f'sentiment_roberta_7subs_{per_sub}per_sub_seed{seed}'
roberta_df = load_parquet_cache(roberta_key, cache_dir=cache_dir)

if roberta_df is None:
    roberta_df = draw_stratified_subset(vader_full, n_per_subreddit=per_sub, seed=seed)
    model_name = cfg['modeling']['sentiment_model']
    batch_size = int(cfg['modeling']['sentiment_batch_size'])
    device = cfg['modeling']['device']
    local_only = bool(cfg['modeling'].get('local_files_only', True))

    print(f'Loading RoBERTa ({model_name}, local_files_only={local_only})...')
    tokenizer, model, resolved_device = load_roberta(
        model_name=model_name,
        device=device,
        local_files_only=local_only,
    )
    print(f'Running genuine RoBERTa on {len(roberta_df):,} comments (device={resolved_device}, batch_size={batch_size})...')
    r_scores = score_roberta(
        texts=roberta_df['cleaned_text'].tolist(),
        tokenizer=tokenizer,
        model=model,
        device=resolved_device,
        batch_size=batch_size,
    )
    roberta_df['negative'] = r_scores['negative'].values
    roberta_df['neutral'] = r_scores['neutral'].values
    roberta_df['positive'] = r_scores['positive'].values
    roberta_df['roberta_label'] = r_scores[['negative', 'neutral', 'positive']].idxmax(axis=1).values
    assert not roberta_df[['negative', 'neutral', 'positive', 'roberta_label']].isna().any().any()
    save_parquet_cache(roberta_df, roberta_key, cache_dir=cache_dir)
else:
    print(f'Loaded cached genuine RoBERTa scores ({roberta_key}, {len(roberta_df):,} rows).')

canonical_parquet = processed_dir / 'sentiment_df.parquet'
roberta_df.to_parquet(canonical_parquet, index=False)
print(f'Canonical scored dataset saved to: {canonical_parquet}')
print('RoBERTa sentiment distribution:')
print(roberta_df['roberta_label'].value_counts(normalize=True).round(4))

## Step 4: RQ1 Model Comparison & Disagreement Analysis
Compute `agreement_rate`, `cohen_kappa`, `macro_f1`, neutral percentages, and inspect the top-5 disagreements.

In [ ]:
comparison = compare_models(roberta_df)

print('=== RQ1: VADER vs. RoBERTa Comparison ===')
print(f"Documents compared : {comparison['n_docs']:,}")
print(f"Disagreements      : {comparison['n_disagreements']:,}")
print(f"Agreement rate     : {comparison['agreement_rate']:.4f} ({comparison['agreement_rate']:.2%})")
print(f"Cohen's kappa      : {comparison['cohen_kappa']:.4f}")
print(f"Macro F1           : {comparison['macro_f1']:.4f}")
print(f"VADER neutral (%)  : {comparison['vader_neutral_pct']:.2f}%")
print(f"RoBERTa neutral (%): {comparison['roberta_neutral_pct']:.2f}%")

print('\nTop-5 Disagreements between VADER and RoBERTa:')
for _, row in comparison['top_disagreements'].head(5).iterrows():
    print(f"[{row['subreddit']}] {str(row['cleaned_text'])[:140]}...")
    print(
        f"  VADER: {row['vader_label']} (comp={row['vader_compound']:.3f}) | "
        f"RoBERTa: {row['roberta_label']} (pos={row['positive']:.3f}, neu={row['neutral']:.3f}, neg={row['negative']:.3f})"
    )

# Export RQ1 tables to data/final/tables/
comparison['confusion_matrix'].to_csv(tables_dir / 'rq1_confusion_matrix.csv')

per_class_rows = []
for cls_name in ['negative', 'neutral', 'positive']:
    m = comparison['per_class'][cls_name]
    per_class_rows.append({
        'class': cls_name,
        'precision_vader_vs_roberta': round(m['precision'], 4),
        'recall_vader_vs_roberta': round(m['recall'], 4),
        'f1_vader_vs_roberta': round(m['f1-score'], 4),
        'support_roberta': int(m['support']),
    })
pd.DataFrame(per_class_rows).to_csv(tables_dir / 'rq1_per_class_agreement.csv', index=False)

rq1_summary = {
    'n_docs': comparison['n_docs'],
    'n_disagreements': comparison['n_disagreements'],
    'agreement_rate': comparison['agreement_rate'],
    'cohen_kappa': comparison['cohen_kappa'],
    'macro_f1': comparison['macro_f1'],
    'vader_neutral_pct': comparison['vader_neutral_pct'],
    'roberta_neutral_pct': comparison['roberta_neutral_pct'],
}
pd.DataFrame([rq1_summary]).to_csv(tables_dir / 'rq1_metrics.csv', index=False)

dis_cols = [
    c for c in ['subreddit', 'created_utc_dt', 'vader_compound', 'vader_label', 'roberta_label', 'negative', 'neutral', 'positive', 'gap', 'cleaned_text']
    if c in comparison['top_disagreements'].columns
]
comparison['top_disagreements'][dis_cols].to_csv(tables_dir / 'rq1_top20_disagreements.csv', index=False)

log_experiment(
    params={'stage': 'rq1_sentiment_validation', 'n_roberta': len(roberta_df), 'n_vader_full': len(vader_full), 'seed': seed},
    metrics=rq1_summary,
    output_dir=exp_dir,
)

## Step 5: VADER vs. RoBERTa Comparison Plot
Save side-by-side correlation scatter and confusion matrix to `data/final/figures/`.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2))
sample_plot = roberta_df.sample(min(2500, len(roberta_df)), random_state=seed)

sns.scatterplot(
    data=sample_plot,
    x='vader_compound',
    y='positive',
    hue='roberta_label',
    palette={'negative': '#d9534f', 'neutral': '#f0ad4e', 'positive': '#5cb85c'},
    alpha=0.45,
    s=16,
    ax=ax1,
)
ax1.set_title(
    f"VADER Compound vs. RoBERTa P(positive)\n(Agreement: {comparison['agreement_rate']:.1%}, kappa={comparison['cohen_kappa']:.3f})",
    fontsize=10.5,
)
ax1.set_xlabel('VADER Compound (-1 to +1)')
ax1.set_ylabel('RoBERTa P(positive)')
ax1.grid(True, linestyle='--', alpha=0.4)

ct_pct = pd.crosstab(roberta_df['vader_label'], roberta_df['roberta_label'], normalize='all') * 100
ct_pct = ct_pct.reindex(index=['negative', 'neutral', 'positive'], columns=['negative', 'neutral', 'positive'], fill_value=0.0)
sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='Blues', cbar=False, linewidths=1, ax=ax2)
ax2.set_title(f'Confusion Matrix (% of {len(roberta_df):,} Comments)', fontsize=10.5)
ax2.set_xlabel('RoBERTa Label (Primary)')
ax2.set_ylabel('VADER Label (Baseline)')

plt.tight_layout()
fig_path = fig_dir / 'vader_vs_roberta_comparison.png'
plt.savefig(fig_path, dpi=300)
plt.show()
print(f'Comparison figure saved to: {fig_path}')